# Fake News Detection & Verification Platform

## NLP Preprocessing, Dataset Splitting & TF-IDF Feature Engineering

This notebook prepares the final cleaned WELFake dataset for machine learning.

### Dataset

Final cleaned WELFake dataset:

`WELFake_Cleanedfinal.csv`

Dataset statistics:

- Total articles: 63,050
- Fake: 34,788
- Real: 28,262
- Fake percentage: 55.14%
- Real percentage: 44.86%

### Objectives

1. Load the final cleaned dataset.
2. Verify dataset integrity.
3. Create reproducible Train/Validation/Test splits.
4. Combine article title and text.
5. Perform NLP preprocessing.
6. Remove linguistic noise.
7. Tokenize the text.
8. Remove English stopwords.
9. Apply lemmatization.
10. Convert text into TF-IDF features.
11. Experiment with unigram and bigram features.
12. Fit TF-IDF only on the training data.
13. Transform validation and test data using the fitted vectorizer.
14. Save the fitted TF-IDF vectorizer for later model training and inference.

### Data Leakage Prevention

The dataset is split before TF-IDF fitting.

The TF-IDF vectorizer is fitted exclusively on the training dataset.

Validation and test datasets are transformed using the already-fitted
training vectorizer.

In [ ]:
import pandas as pd
import numpy as np

import re
import os
import pickle
import time
import csv

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

import nltk

In [ ]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [ ]:
df = pd.read_csv(
    "WELFake_Cleanedfinal.csv"
)

In [ ]:
print("=" * 50)
print("DATASET VERIFICATION")
print("=" * 50)

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nLabel percentages:")
print(
    df["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

DATASET VERIFICATION

Shape:
(63050, 5)

Columns:
['Unnamed: 0', 'title', 'text', 'label', 'combined_text']

Missing values:
Unnamed: 0         0
title            519
text               0
label              0
combined_text      0
dtype: int64

Duplicate rows:
0

Label distribution:
label
0    34788
1    28262
Name: count, dtype: int64

Label percentages:
label
0    55.18
1    44.82
Name: proportion, dtype: float64


In [ ]:
label_mapping = {
    0: "Fake",
    1: "Real"
}

df["label_name"] = df["label"].map(label_mapping)

In [ ]:
df["label_name"].value_counts()

,count
label_name,
Fake,34788
Real,28262


In [ ]:
sample = df.sample(
    1,
    random_state=42
).iloc[0]

print("TITLE:")
print(sample["title"])

print("\nARTICLE TEXT:")
print(sample["text"][:2000])

print("\nCOMBINED TEXT:")
print(sample["combined_text"][:2500])

TITLE:
Australia to hunt down anti-vax nurses and prosecute them for disobeying the medical police state

ARTICLE TEXT:
Australia to hunt down anti-vax nurses and prosecute them for disobeying the medical police state Vicki Batts Tags: vaccination , Australia , medical police state (NaturalNews) Is there some sort of race to see which country can eliminate the rights of its people first? It is certainly beginning to feel like there must be something going on, since government overreach looks like it is reaching an all-time high across the world.While countries like Australia demonize other nations for their lack of progressiveness, recent developments suggest that its government is taking away people's freedom to think for themselves, slowly but surely chipping away at those who have dissenting opinions. The evidence? The newly released vaccination standards provided by The Nursing and Midwifery Board of Australia in response to what the organization described as, "a small number of nu

In [ ]:
X = df["combined_text"]
y = df["label"]

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [ ]:
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

print(
    "\nTotal:",
    len(X_train) +
    len(X_val) +
    len(X_test)
)

Training: 50440
Validation: 6305
Testing: 6305

Total: 63050


In [ ]:
print("TRAIN")
print(y_train.value_counts())
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nVALIDATION")
print(y_val.value_counts())
print(
    y_val.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTEST")
print(y_test.value_counts())
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

TRAIN
label
0    27830
1    22610
Name: count, dtype: int64
label
0    55.17
1    44.83
Name: proportion, dtype: float64

VALIDATION
label
0    3479
1    2826
Name: count, dtype: int64
label
0    55.18
1    44.82
Name: proportion, dtype: float64

TEST
label
0    3479
1    2826
Name: count, dtype: int64
label
0    55.18
1    44.82
Name: proportion, dtype: float64


In [ ]:
def preprocess_text(text):
    text = str(text)

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+|https\S+",
        " ",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<.*?>",
        " ",
        text
    )

    # Keep alphabetic characters
    text = re.sub(
        r"[^a-z\s]",
        " ",
        text
    )

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    # Tokenization
    tokens = text.split()

    # Stopword removal
    tokens = [
        word
        for word in tokens
        if word not in stop_words
    ]

    # Lemmatization
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]

    return " ".join(tokens)

In [ ]:
sample_text = X_train.iloc[0]

processed_sample = preprocess_text(
    sample_text
)

print("ORIGINAL:")
print(sample_text[:2000])

print("\n" + "=" * 80)

print("PROCESSED:")
print(processed_sample[:2000])

ORIGINAL:
Ammon Bundy’s Lunatic Militia Men Facing More Charges From The Feds The saga surrounding the idiots who decided to invade and occupy a federal building on a wildlife preserve in Oregon continues. Though the occupation itself is over, the legal troubles for this group of right-wing loons are just beginning. It has been reported that Bundy and his band of nutbags have been indicted on yet more federal charges.These latest indictments state that they will be facing charges because they had weapons in federal facilities, as well as with vandalism and theft of property belonging to the government. The original federal charges drop the hammer on the group 26 people charged in all for conspiracy to disrupt the duties of federal agents. Also, their infamous stealing of a government truck for a grocery store trip is included in these latest indictments. There are also charges of using weapons during violent criminal activity.Another blow came to the group when the death of LaVoy Finic

In [ ]:
start_time = time.time()

X_train_processed = X_train.apply(
    preprocess_text
)

end_time = time.time()

print(
    f"Training preprocessing time: "
    f"{end_time - start_time:.2f} seconds"
)

Training preprocessing time: 107.87 seconds


In [ ]:
start_time = time.time()

X_val_processed = X_val.apply(
    preprocess_text
)

end_time = time.time()

print(
    f"Validation preprocessing time: "
    f"{end_time - start_time:.2f} seconds"
)

Validation preprocessing time: 20.83 seconds


In [ ]:
start_time = time.time()

X_test_processed = X_test.apply(
    preprocess_text
)

end_time = time.time()

print(
    f"Test preprocessing time: "
    f"{end_time - start_time:.2f} seconds"
)

Test preprocessing time: 14.58 seconds


In [ ]:
print(X_train_processed.head())

28881    ammon bundy lunatic militia men facing charge ...
43623    november daily contrarian read november daily ...
55540    obama clock boy come back texas spending month...
18083    president obama make fun donald trump hilariou...
39834    mexico prison population drop police prosecuto...
Name: combined_text, dtype: object


In [ ]:
print(
    "Training samples:",
    len(X_train_processed)
)

print(
    "Validation samples:",
    len(X_val_processed)
)

print(
    "Testing samples:",
    len(X_test_processed)
)

Training samples: 50440
Validation samples: 6305
Testing samples: 6305


In [ ]:
print(
    "Empty training samples:",
    (
        X_train_processed.str.strip() == ""
    ).sum()
)

print(
    "Empty validation samples:",
    (
        X_val_processed.str.strip() == ""
    ).sum()
)

print(
    "Empty test samples:",
    (
        X_test_processed.str.strip() == ""
    ).sum()
)

Empty training samples: 35
Empty validation samples: 2
Empty test samples: 8


In [ ]:
train_original_words = (
    X_train.str.split().str.len()
)

train_processed_words = (
    X_train_processed.str.split().str.len()
)

In [ ]:
print(
    "Average original words:",
    train_original_words.mean()
)

print(
    "Average processed words:",
    train_processed_words.mean()
)

Average original words: 558.0908802537668
Average processed words: 316.26338223632035


In [ ]:
reduction = (
    1 -
    train_processed_words.mean()
    /
    train_original_words.mean()
) * 100

print(
    f"Average word reduction: {reduction:.2f}%"
)

Average word reduction: 43.33%


In [ ]:
tfidf = TfidfVectorizer(
    max_features=60000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

In [ ]:
start_time = time.time()

X_train_tfidf = tfidf.fit_transform(
    X_train_processed
)

end_time = time.time()

print(
    f"TF-IDF training time: "
    f"{end_time - start_time:.2f} seconds"
)

TF-IDF training time: 93.98 seconds


In [ ]:
print(
    "Training TF-IDF shape:",
    X_train_tfidf.shape
)

Training TF-IDF shape: (50440, 60000)


In [ ]:
X_val_tfidf = tfidf.transform(
    X_val_processed
)

In [ ]:
print(
    "Validation TF-IDF shape:",
    X_val_tfidf.shape
)

Validation TF-IDF shape: (6305, 60000)


In [ ]:
X_test_tfidf = tfidf.transform(
    X_test_processed
)

In [ ]:
print(
    "Test TF-IDF shape:",
    X_test_tfidf.shape
)

Test TF-IDF shape: (6305, 60000)


In [ ]:
print("Train features:", X_train_tfidf.shape[1])
print("Validation features:", X_val_tfidf.shape[1])
print("Test features:", X_test_tfidf.shape[1])

Train features: 60000
Validation features: 60000
Test features: 60000


In [ ]:
feature_names = tfidf.get_feature_names_out()

print(
    "Vocabulary size:",
    len(feature_names)
)

print("\nFirst 100 features:")
print(feature_names[:100])

Vocabulary size: 60000

First 100 features:
['aa' 'aaa' 'aap' 'aapl' 'aaron' 'aaron klein' 'aaronkleinshow'
 'aaronkleinshow follow' 'aarp' 'ab' 'aba' 'abaaoud' 'ababa' 'aback'
 'abadi' 'abadi said' 'abandon' 'abandoned' 'abandoning' 'abandonment'
 'abbas' 'abbey' 'abbott' 'abbott said' 'abby' 'abc' 'abc cbs' 'abc good'
 'abc news' 'abc week' 'abcpolitics' 'abd' 'abd rabbu' 'abdel'
 'abdel fattah' 'abdeslam' 'abdicate' 'abdication' 'abdomen' 'abducted'
 'abduction' 'abdul' 'abdulazeez' 'abdulaziz' 'abdullah' 'abdullah saleh'
 'abe' 'abe japan' 'abe said' 'abedi' 'abedin' 'abedin email'
 'abedin weiner' 'abedini' 'aber' 'aberdeen' 'abetting' 'abfalecbaldwin'
 'abfoundation' 'abfoundation abfalecbaldwin' 'abhorrent' 'abid' 'abide'
 'abiding' 'abiding citizen' 'abidjan' 'abigail' 'ability'
 'ability deliver' 'ability get' 'ability make' 'abject' 'ablaze' 'able'
 'able afford' 'able buy' 'able carry' 'able come' 'able continue'
 'able control' 'able find' 'able get' 'able go' 'able handle'

In [ ]:
print(
    "Training matrix shape:",
    X_train_tfidf.shape
)

print(
    "Non-zero values:",
    X_train_tfidf.nnz
)

total_elements = (
    X_train_tfidf.shape[0] *
    X_train_tfidf.shape[1]
)

sparsity = (
    1 -
    X_train_tfidf.nnz /
    total_elements
) * 100

print(
    f"Sparsity: {sparsity:.2f}%"
)

Training matrix shape: (50440, 60000)
Non-zero values: 12934446
Sparsity: 99.57%


In [ ]:
os.makedirs(
    "models",
    exist_ok=True
)

In [ ]:
with open(
    "models/vectorizer.pkl",
    "wb"
) as file:

    pickle.dump(
        tfidf,
        file
    )

In [ ]:
print(
    "TF-IDF vectorizer saved successfully."
)

TF-IDF vectorizer saved successfully.


In [ ]:
train_processed = pd.DataFrame({
    "text": X_train_processed,
    "label": y_train.values
})

validation_processed = pd.DataFrame({
    "text": X_val_processed,
    "label": y_val.values
})

test_processed = pd.DataFrame({
    "text": X_test_processed,
    "label": y_test.values
})

In [ ]:
train_processed.to_csv(
    "train_processed.csv",
    index=False
)

validation_processed.to_csv(
    "validation_processed.csv",
    index=False
)

test_processed.to_csv(
    "test_processed.csv",
    index=False
)

# Notebook 2 Summary

The final cleaned WELFake dataset was divided into:

- Training set: 80%
- Validation set: 10%
- Testing set: 10%

The split was performed using stratified sampling with a fixed random seed
of 42 to maintain approximately the same Fake/Real class distribution
across all subsets.

The text was processed using:

1. Lowercasing
2. URL removal
3. HTML removal
4. Non-alphabetic character removal
5. Whitespace normalization
6. Tokenization
7. Stopword removal
8. Lemmatization

The title and article body were combined before NLP processing.

TF-IDF features were generated using:

- Unigrams and bigrams
- Maximum 60,000 features
- Minimum document frequency of 2
- Maximum document frequency of 95%
- Sublinear term-frequency scaling

The TF-IDF vectorizer was fitted exclusively on the training data.

Validation and test data were transformed using the fitted training
vectorizer to prevent data leakage.

The fitted vectorizer was saved for subsequent model training and
production inference.